## Tâche 1 : Initialisation et Lecture (Zone Bronze) 
## 1. Configurez votre session Spark pour vous connecter à MinIO. 



In [29]:
from pyspark.sql import SparkSession

In [30]:
# Configuration de la session Spark
spark = SparkSession.builder \
    .appName("TP2-HelloDataLake") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-datalake:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Session Spark initialisée avec succès !")

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
def display(df, n=20)

## 2. Créez un bucket nommé gold et un bucket nommé silver dans l'interface de MinIO. 

## Done

## 3. Chargez le fichier CSV des taxis dans un DataFrame Spark en activant l'inférence de schéma (inferSchema) et les en-têtes (header).

In [3]:
# Lecture du fichier CSV
path_csv = "s3a://bronze/taxis/yellow_tripdata_2020-01.csv"

In [5]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv(path_csv)

##  Affichez le schéma des données. 

In [28]:
# Affichage des premières lignes
df.show(10, False).T

ConnectionRefusedError: [Errno 111] Connection refused

## Tâche 2 : Nettoyage des données (Zone Silver) Le dataset contient des erreurs de saisie et des valeurs impossibles. Filtrez le DataFrame pour ne garder que les trajets validés selon ces critères :    

In [13]:
from pyspark.sql.functions import col


## 1. Le nombre de passagers (passenger_count) doit être strictement supérieur à 0.

In [14]:
# Filtrer les lignes où la colonne est nulle
df_nuls_1 = df.filter(col("passenger_count").isNull())
df_nuls_1.show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|    NULL| 2020-01-01 08:51:00|  2020-01-01 09:19:00|           NULL|        13.69|      NULL|              NULL|         136|         232|        NULL|      51.05| 2.75|    0.5|       0.0|         0.0|                  0.3

In [15]:
print(f"Le nombre de valeurs qui sont nulls avec passenger_count sont: {df_nuls_1.count()}")

Le nombre de valeurs qui sont nulls avec passenger_count sont: 65441


In [16]:
# On filtre pour garder uniquement les trajets avec plus de 0 passager
df_filtre_1 = df.filter(df.passenger_count > 0)

#  vérifions combien de lignes il reste
print("Nombre de lignes après le filtre 1 :", df_filtre_1.count())

Nombre de lignes après le filtre 1 : 6225265


## 2. La distance du trajet (trip_distance) doit être strictement supérieure à 0.

In [17]:
# On applique le deuxième filtre sur la distance
df_filtre_2 = df_filtre_1.filter(df_filtre_1.trip_distance > 0)

# Voyons l'évolution du nombre de lignes
print("Nombre de lignes après le filtre 2 :", df_filtre_2.count())

Nombre de lignes après le filtre 2 : 6159922


## 3. Le montant de la course (fare_amount) doit être positif.

In [18]:
# On applique le troisième filtre sur le montant de la course
df_filtre_3 = df_filtre_2.filter(df_filtre_2.fare_amount >= 0)

# Verifions
print("Nombre de lignes après le filtre 3 :", df_filtre_3.count())

Nombre de lignes après le filtre 3 : 6142818


## 4. Supprimez les lignes qui contiennent des valeurs manquantes (Null) dans les colonnes pickup_longitude ou dropoff_longitude (si présentes, sinon sur payment_type). 

In [18]:
df_nuls_2 = df_filtre_3.filter(col("PULocationID").isNull())
print(df_nuls_2.count())

0


In [19]:
df_nuls_3 = df_filtre_3.filter(col("DOLocationID").isNull())
print(df_nuls_3.count())

0


In [ ]:
# df_filtre_3.filter(col("dropoff_longitude").isNull())

In [20]:
df_nuls_4 = df_filtre_3.filter(col("payment_type").isNull())
print(df_nuls_4.count())

0


In [20]:
# On supprime les lignes où payment_type est Null
df_silver_final = df_filtre_3.dropna(subset=["payment_type"])

# On affiche un aperçu de notre DataFrame propre
df_silver_final.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2020-01-01 00:28:15|  2020-01-01 00:33:03|              1|          1.2|         1|                 N|         238|         239|           1|        6.0|  3.0|    0.5|      1.47|         0.0|                  0.3

In [ ]:
df_silver_final.write.mode("overwrite").parquet("s3a://silver/tpmilia/countries/")

## Tâche 3 : Transformations et Calculs 
## 1. Créez une nouvelle colonne nommée total_price_paid qui est la somme de fare_amount, tip_amount et tolls_amount. 

 

In [21]:
from pyspark.sql.functions import col

# Création de la colonne total_price_paid
df_transfo_1 = df_silver_final.withColumn(
    "total_price_paid", 
    col("fare_amount") + col("tip_amount") + col("tolls_amount")
)

# On vérifie que la nouvelle colonne est bien là
df_transfo_1.select("fare_amount", "tip_amount", "tolls_amount", "total_price_paid").show(5)

+-----------+----------+------------+----------------+
|fare_amount|tip_amount|tolls_amount|total_price_paid|
+-----------+----------+------------+----------------+
|        6.0|      1.47|         0.0|            7.47|
|        7.0|       1.5|         0.0|             8.5|
|        6.0|       1.0|         0.0|             7.0|
|        5.5|      1.36|         0.0|            6.86|
|        2.5|       0.0|         0.0|             2.5|
+-----------+----------+------------+----------------+
only showing top 5 rows



## 2. Créez une colonne tip_percentage qui calcule la part du pourboire par rapport au montant de la course (fare_amount). Arrondissez à deux décimales. 

In [22]:
from pyspark.sql.functions import round

# Calcul du pourcentage de pourboire arrondi à 2 décimales
df_transfo_2 = df_transfo_1.withColumn(
    "tip_percentage", 
    round((col("tip_amount") / col("fare_amount")) * 100, 2)
)

#  On vérifie le résultat
df_transfo_2.select("fare_amount", "tip_amount", "tip_percentage").show(5)

+-----------+----------+--------------+
|fare_amount|tip_amount|tip_percentage|
+-----------+----------+--------------+
|        6.0|      1.47|          24.5|
|        7.0|       1.5|         21.43|
|        6.0|       1.0|         16.67|
|        5.5|      1.36|         24.73|
|        2.5|       0.0|           0.0|
+-----------+----------+--------------+
only showing top 5 rows



## 3. À l'aide des fonctions Spark (ou de la fonction hour()), extrayez l'heure de la colonne tpep_pickup_datetime dans une nouvelle colonne pickup_hour.

In [23]:
from pyspark.sql.functions import hour

# Extraction de l'heure de départ
df_transfo_3 = df_transfo_2.withColumn(
    "pickup_hour", 
    hour(col("tpep_pickup_datetime"))
)

# On vérifie que l'heure est bien extraite
df_transfo_3.select("tpep_pickup_datetime", "pickup_hour").show(5)

+--------------------+-----------+
|tpep_pickup_datetime|pickup_hour|
+--------------------+-----------+
| 2020-01-01 00:28:15|          0|
| 2020-01-01 00:35:39|          0|
| 2020-01-01 00:47:41|          0|
| 2020-01-01 00:55:23|          0|
| 2020-01-01 00:09:44|          0|
+--------------------+-----------+
only showing top 5 rows



## Tâche 4 : Partitionnement et Stockage (Vers MinIO Silver) 
## 1. Sauvegardez votre DataFrame nettoyé dans le bucket s3a://silver/taxis/yellow_clean/.


In [25]:
target_path = "s3a://silver/taxis/yellow_clean/"

In [24]:
# Sauvegarde au format Parquet avec partitionnement
df_transfo_3.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("passenger_count") \
    .save("s3a://silver/taxis/yellow_clean/")

print("Données sauvegardées avec succès dans la zone Silver !")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_com

Py4JError: An error occurred while calling o119.save

In [27]:
print("Enregistrement des données partitionnées par Continent dans MinIO...")

df_transfo_3.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("passenger_count") \
    .parquet(target_path)

print("Données sauvegardées avec succès au format Parquet !")

Enregistrement des données partitionnées par Continent dans MinIO...


ConnectionRefusedError: [Errno 111] Connection refused

## 2. Utilisez le format Parquet et partitionnez les données par la colonne passenger_count (pour optimiser les recherches futures selon le nombre de passagers). 

## Tâche 5 : Analyse SQL (Zone Gold) 
## 1. Enregistrez votre DataFrame nettoyé en tant que vue SQL temporaire. 


In [26]:
# On crée une vue SQL temporaire nommée "v_taxis_clean"
df_transfo_3.createOrReplaceTempView("v_taxis_clean")

print("Vue SQL temporaire créée !")

ConnectionRefusedError: [Errno 111] Connection refused

## 2. Écrivez une requête SQL pour trouver le pourboire moyen (tip_amount) et la distance moyenne en fonction de l'heure de la journée (pickup_hour ou selon le type de paiement payment_type). 
 

## 3. Écrivez le résultat final de cette analyse au format Parquet dans s3a://gold/taxis/hourly_stats/.